In [1]:
from spin_lattices import KagomeLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from boolean_analysis import BooleanFourierAnalyser, keep_largest_n, ScorerType, get_scorer
from pathlib import Path
import numpy as np
import pandas as pd
import lattice_symmetries as ls
import matplotlib.pyplot as plt
from heisenberg_hamiltonians import batched_state_info_df
from itertools import product
import numpy.typing as npt
from tqdm import tqdm
import seaborn as sns

%matplotlib widget

2023-01-18 15:29:33.486 | DEBUG    | lattice_symmetries:__init__:49 - Initializing Haskell runtime...
2023-01-18 15:29:33.491 | DEBUG    | lattice_symmetries:__init__:51 - Initializing Chapel runtime...
[Debug]   [2023-01-18 15:29:33.527 | DEBUG    | lattice_symmetries:__init__:53 - Setting Python exception handler...
LOCALE0]   Initializing chpl_kernels ...
set_python_exception_handler ...


## How properties of Fourier spectra changes with the lattice size
1. How many elements of Fourier spectra are needed to achieve 95% overlap?
2. How fast the absolute value of Fourier coefficients decline comparing with the coefficients of the ground state in the ordinary basis.
3. What happens if we keep elements of small degrees?
3. Which probability distribution is better to use?

In [4]:
J2 = 0.9
system = HeisenbergJ1J2(
    KagomeLattice(width=2, height=2),
    J1=1,
    J2=J2,
    use_symmetries=True,
    spin_inversion=1,
    ground_state_cache_dir=Path("groundstates"),
)
analyzer = BooleanFourierAnalyser(
    system=system,
    use_subset_symmetries=True,
)


number_spins=12
Symmetry group contains 16 elements
Hilbert space dimension is 47
Finding system ground state
Calculating eigenvalues / eigenstates


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


Ground state energy is -20.6362854929


In [33]:
def how_many_terms_to_achieve_score(
    analyzer: BooleanFourierAnalyser,
    scorer: str | ScorerType = "overlap",
    target_score: float = 0.95,
    min_terms: int = 1,
    max_terms: int = 101,
    step: int = 10,
    show_progress: bool = True,
    batch_size=100,
    max_train_set_size=10000,
    weighted_sampling: bool = False,
) -> int | None:
    """How many terms are needed to reconstruct the sign structure of the ground state
    with the given score?

    Parameters
    ----------
    analyzer : BooleanFourierAnalyser
        The analyzer to use.

    scorer : str | ScorerType, optional

    target_score : float, optional
        The target score, by default 0.95

    min_terms : int, optional
        The minimum number of terms to try, by default 1

    max_terms : int, optional
        The maximum number of terms to try, by default 101

    step : int, optional
        The step size, by default 10

    Returns
    -------
    int | None
        The number of terms needed to achieve the target score, or None if the target
        score could not be achieved.
    """
    train_set = analyzer.system.canonical_basis.states
    if len(train_set) > max_train_set_size:
        train_set = np.random.choice(
            train_set,
            max_train_set_size,
            replace=False,
            p=(
                analyzer.system.get_df_ground_state(canonical_basis=True).loc[
                    train_set, "amplitude"
                ]
                ** 2
            ).values
            if weighted_sampling
            else None,
        )

    analyzer.fit(train_set, batch_size=batch_size)

    evaluation_set = analyzer.system.basis.states
    prediction = np.zeros_like(evaluation_set, dtype="float64")
    state_info_df = batched_state_info_df(
        analyzer.system.basis, analyzer.system.canonical_basis.states
    )

    for n_terms in range(min_terms, max_terms, step):
        if n_terms == min_terms:
            analyzer.set_truncate_strategy(keep_largest_n(n_terms))
        else:
            analyzer.set_truncate_strategy(keep_largest_n(step, offset=max(0, n_terms - step)))

        prediction += analyzer.predict(evaluation_set)

        true = analyzer._get_signal_df(analyzer.system.canonical_basis.states)
        prediction_expanded = np.asarray(
            state_info_df.merge(
                pd.Series(prediction, name="prediction", index=evaluation_set),
                left_on="representative",
                right_index=True,
                how="left",
            )["prediction"].values
        )
        score = get_scorer(scorer)(true, np.sign(prediction_expanded))
        if show_progress:
            print(f"{n_terms} terms: {score:.3f}")
        if score >= target_score:
            return n_terms
    return None


In [8]:
# Marshall

analyzer = BooleanFourierAnalyser(
    system=HeisenbergJ1J2(
        KagomeLattice(width=2, height=3),
        J1=1,
        J2=0,
        use_symmetries=True,
        spin_inversion=1,
        ground_state_cache_dir=Path("groundstates"),
    ),
    use_subset_symmetries=True,
)
assert how_many_terms_to_achieve_score(analyzer, scorer="accuracy", target_score=0.99) == 1
assert how_many_terms_to_achieve_score(analyzer, scorer="overlap", target_score=0.99) == 1
assert how_many_terms_to_achieve_score(analyzer, scorer="accuracy", target_score=1.01) is None

# Frustrated

analyzer = BooleanFourierAnalyser(
    system=HeisenbergJ1J2(
        KagomeLattice(width=2, height=3),
        J1=1,
        J2=0.8,
        use_symmetries=True,
        spin_inversion=1,
        ground_state_cache_dir=Path("groundstates"),
    ),
    use_subset_symmetries=True,
)


def assert_is_large(terms: int | None):
    if terms is not None:
        assert terms > 1


assert_is_large(how_many_terms_to_achieve_score(analyzer, scorer="accuracy", target_score=0.99))
assert_is_large(how_many_terms_to_achieve_score(analyzer, scorer="overlap", target_score=0.99))


number_spins=18
Symmetry group contains 12 elements
Hilbert space dimension is 2102


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


Finding system ground state
Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x3-1.0-0.0-True-1-1.pickle
Ground state energy is -34.6339591099


  0%|          | 0/10 [00:00<?, ?it/s]


1 terms: 1.000


...

31 terms: 0.967
41 terms: 0.982
51 terms: 0.981


 80%|████████  | 8/10 [00:00<00:00, 10.91it/s]

61 terms: 0.975
71 terms: 0.965
81 terms: 0.958


100%|██████████| 10/10 [00:00<00:00, 10.60it/s]

91 terms: 0.958


In [11]:
results = []
for J2 in tqdm([0.5, 0.6, 0.7, 0.8, 0.9, 1]):
    analyzer = BooleanFourierAnalyser(
        system=HeisenbergJ1J2(
            KagomeLattice(width=2, height=4),
            J1=1,
            J2=J2,
            use_symmetries=True,
            spin_inversion=1,
            ground_state_cache_dir=Path("groundstates"),
        ),
        use_subset_symmetries=True,
        show_progress=True,
    )
    terms = how_many_terms_to_achieve_score(
        analyzer,
        scorer="overlap",
        target_score=0.95,
        step=10,
        max_terms=1001,
        show_progress=True,
        batch_size=1000,
    )
    results.append(dict(J2=J2, terms=terms))
results_df = pd.DataFrame(results)
sns.relplot(data=results_df, x="J2", y="terms", kind="line")


  0%|          | 0/6 [00:00<?, ?it/s][Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


number_spins=24
Symmetry group contains 16 elements
Hilbert space dimension is 85662


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


Finding system ground state
Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-0.5-True-1-1.pickle
Ground state energy is -39.4742602743


Batch 1/10


...

141 terms: 0.286


151 terms: 0.310


161 terms: 0.313


 17%|█▋        | 1/6 [05:11<25:58, 311.68s/it]


KeyboardInterrupt: 

In [60]:
system.canonical_basis.states.shape


(924,)

In [62]:
results_df


,J2,terms
0,0.00,1
1,0.30,1
2,0.50,1
3,0.55,1
4,0.60,1
5,0.70,1
6,0.80,1
7,0.90,1
8,1.00,1


In [63]:
analyzer = BooleanFourierAnalyser(
    system=HeisenbergJ1J2(
        KagomeLattice(width=2, height=2),
        J1=1,
        J2=0.8,
        use_symmetries=True,
        spin_inversion=1,
        ground_state_cache_dir=Path("groundstates"),
    ),
    use_subset_symmetries=True,
)


number_spins=12
Symmetry group contains 16 elements
Hilbert space dimension is 47
Finding system ground state
Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x2-1.0-0.8-True-1-1.pickle
Ground state energy is -20.0382298203


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


In [66]:
analyzer.fit(analyzer.system.canonical_basis.states)
analyzer.set_truncate_strategy(keep_largest_n(1)).prediction_score(
    analyzer.system.canonical_basis.states
)


-0.2537071402732067

In [69]:
how_many_terms_to_achieve_score(analyzer)


0.9941477861378842


1

In [36]:
analyzer = BooleanFourierAnalyser(
    system=HeisenbergJ1J2(
        KagomeLattice(width=2, height=3),
        J1=1,
        J2=0.8,
        use_symmetries=True,
        spin_inversion=1,
        ground_state_cache_dir=Path("groundstates"),
    ),
    use_subset_symmetries=True,
)

assert_is_large(
    how_many_terms_to_achieve_score(
        analyzer, scorer="accuracy", target_score=0.95, max_train_set_size=1000,
        max_terms=10000, weighted_sampling=True
    )
)


number_spins=18
Symmetry group contains 12 elements
Hilbert space dimension is 2102


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


Finding system ground state
Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x3-1.0-0.8-True-1-1.pickle
Ground state energy is -29.4963743253
1 terms: 0.522
11 terms: 0.633
21 terms: 0.654
31 terms: 0.665
41 terms: 0.659
51 terms: 0.669
61 terms: 0.682
71 terms: 0.690
81 terms: 0.670
91 terms: 0.677
101 terms: 0.673
111 terms: 0.678
121 terms: 0.670
131 terms: 0.659
141 terms: 0.670
151 terms: 0.658
161 terms: 0.648
171 terms: 0.646
181 terms: 0.655
191 terms: 0.651
201 terms: 0.638
211 terms: 0.633
221 terms: 0.625
231 terms: 0.629
241 terms: 0.623
251 terms: 0.625
261 terms: 0.616
271 terms: 0.626
281 terms: 0.627
291 terms: 0.617
301 terms: 0.615
311 terms: 0.625
321 terms: 0.631
331 terms: 0.622
341 terms: 0.617
351 terms: 0.620
361 terms: 0.616
371 terms: 0.622
381 terms: 0.614
391 terms: 0.611
401 terms: 0.615
411 terms: 0.609
421 terms: 0.617
431 terms: 0.615
441 terms: 0.610
451 terms: 0.612
461 terms: 0.612
471 terms: 0.602
481 t

KeyboardInterrupt: 

(2102,)

In [21]:
evaluation_set = analyzer.system.basis.states
state_info_df = batched_state_info_df(
    analyzer.system.basis, analyzer.system.canonical_basis.states
)
scorer = "overlap"

prediction = analyzer.set_truncate_strategy(keep_largest_n(91)).predict(evaluation_set)

true = analyzer._get_signal_df(analyzer.system.canonical_basis.states)
prediction_expanded = np.asarray(
    state_info_df.merge(
        pd.Series(prediction, name="prediction", index=evaluation_set),
        left_on="representative",
        right_index=True,
        how="left",
    )["prediction"].values
)
score = get_scorer(scorer)(true, np.sign(prediction_expanded))
score


0.8798721610949767

In [38]:
import lzma
import pickle
from itertools import product
from pathlib import Path

from tqdm import tqdm

from boolean_analysis import BooleanFourierAnalyser
from heisenberg_hamiltonians import HeisenbergJ1J2
from spin_lattices import KagomeLattice

batch_size = 1000
experiment_dir = Path("experiments/kagome24-2023-01-18")
experiment_dir.mkdir(exist_ok=True, parents=True)

for J2 in tqdm([0.5, 0.6, 0.7, 0.8, 0.9, 1, 0.55, 0.65, 0.75, 0.85, 0.95][:1]):
    pickle_to = f"learner-J2={J2!r}.pickle"

    analyzer = BooleanFourierAnalyser(
        system=HeisenbergJ1J2(
            KagomeLattice(width=2, height=4),
            J1=1,
            J2=J2,
            use_symmetries=True,
            spin_inversion=1,
            ground_state_cache_dir=Path("groundstates"),
        ),
        use_subset_symmetries=True,
        show_progress=True,
    )
    train_set = analyzer.system.canonical_basis.states[:1000]
    analyzer.fit(train_set, batch_size=batch_size)
    with lzma.open(experiment_dir / pickle_to, "wb") as f:
        pickle.dump(analyzer.learner, f)


  0%|          | 0/1 [00:00<?, ?it/s][Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


number_spins=24
Symmetry group contains 16 elements
Hilbert space dimension is 85662


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


Finding system ground state
Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-0.5-True-1-1.pickle
Ground state energy is -39.4742602743


Batch 1/1


100%|██████████| 1/1 [00:25<00:00, 25.57s/it]


In [41]:
analyzer.learner.subsets.shape

(528656,)

In [42]:
528656 * 2

1057312

In [44]:
dir(analyzer.learner)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_ensure_fitted',
 'coeffs_',
 'fit',
 'get_coeffs_ser',
 'number_spins',
 'subsets',
 'x_',
 'y_']

In [49]:
analyzer.learner.coeffs_.nbytes

4229248

In [50]:
analyzer.learner

array([ 4095,  6143,  7167,  7679,  7935,  8063,  8127,  8159,  8175,
        8183,  8187,  8189,  8190, 10239, 11263, 11775, 12031, 12159,
       12223, 12255, 12271, 12279, 12283, 12285, 12286, 13311, 13823,
       14079, 14207, 14271, 14303, 14319, 14327, 14331, 14333, 14334,
       14847, 15103, 15231, 15295, 15327, 15343, 15351, 15355, 15357,
       15358, 15615, 15743, 15807, 15839, 15855, 15863, 15867, 15869,
       15870, 15999, 16063, 16095, 16111, 16119, 16123, 16125, 16126,
       16191, 16223, 16239, 16247, 16251, 16253, 16254, 16287, 16303,
       16311, 16315, 16317, 16318, 16335, 16343, 16347, 16349, 16350,
       16359, 16363, 16365, 16366, 16371, 16373, 16374, 16377, 16378,
       16380, 18431, 19455, 19967, 20223, 20351, 20415, 20447, 20463,
       20471, 20475, 20477, 20478, 21503, 22015, 22271, 22399, 22463,
       22495, 22511, 22519, 22523, 22525, 22526, 23039, 23295, 23423,
       23487, 23519, 23535, 23543, 23547, 23549, 23550, 23807, 23935,
       23999, 24031,